# JetRacer - Tune bam vach truc tiep tren JupyterLab

Notebook nay thay cho viec SSH vao Jetson go lenh. Mo tu may khac, keo slider, nhin ngay ket qua tren dung frame camera dang chay.

**Truoc moi lan Run All**: chon `Kernel > Restart & Clear Output` de kernel cu release camera. Hai notebook mo camera cung luc se bao `Failed to create CaptureSession`.

**An toan**: mac dinh `driver_kind='dryrun'` - banh KHONG quay du keo slider the nao. Chi doi sang `'nvidia'` o cell cuoi khi da san sang cho xe chay.

## 1. Kiem tra thu muc va file

In [ ]:
%cd /home/jetson/JetsonRacer

import os
import socket

required = [
    'tools/tune_lane_jupyter.py',
    'src/jetracer_baseline/tuning_ui.py',
    'src/jetracer_baseline/perception/lane.py',
    'src/jetracer_baseline/perception/shading.py',
    'configs/default.yaml',
]
missing = [p for p in required if not os.path.exists(p)]
print('Hostname:', socket.gethostname())
print('Thu muc:', os.getcwd())
print('Files:', 'OK' if not missing else 'THIEU ' + ', '.join(missing))
if missing:
    raise IOError('Chua copy du code moi sang Jetson')

print('Shading da hieu chuan:', os.path.exists('configs/shading.yaml'))

## 2. Mo giao dien (che do AN TOAN - banh khong quay)

Thu tu thao tac:

1. Bam **1. MO CAMERA**, doi preview hien ra.
2. Chon **Mau vach** dung sa ban: `do` cho sa ban tap, `trang` cho sa ban thi.
3. Nhin o **MASK (nguong mau)** goc tren phai. Muc tieu: chi con vach, khong con dom nhieu, khong dinh vien lane.
4. Keo slider trong tab **Bam vach** cho den khi mask sach VA `dai` trong banner >= 4.
5. Xem o **BIRD'S-EYE**: duong do phai nam dung tren vach, cham tim la diem ngam.
6. Doi thay xe di, nhin do thi **CTE/LAI** goc duoi phai. Duong xanh rang cua lien tuc = dang dao dong -> sang tab **Dieu khien** giam `PID Kp`.
7. Bam **LUU CONFIG** khi da vua y.

In [ ]:
%run tools/tune_lane_jupyter.py

## 3. Chan doan nhanh

| Nhin thay | Nguyen nhan | Keo slider nao |
|---|---|---|
| Mask day dom trang | Nguong qua rong | Tang `Bao hoa toi thieu (S)` hoac `Dien tich blob toi thieu` |
| Mask trong, banner bao MAT VACH | Nguong qua chat | Giam `Bao hoa toi thieu (S)`, giam `Dien tich blob toi thieu` |
| Duong do bam vao vien lane | Bat nham vet rong | Giam `Be rong cum toi da` |
| `dai` chi 1-2 | It du lieu, sap mat vach | Giam `Dien tich blob toi thieu` va `Pixel toi thieu / dai` |
| Nen van/tran lot vao mask | ROI cat chua du sau | Tang `ROI tren` |
| Duong xanh rang cua | Lai dao dong | Giam `PID Kp`, giam `Trong so diem ngam` |
| Xe cat cua muon | Sua qua muon | Tang `Trong so diem ngam` |
| Ga luon o muc thap nhat | Bo ga qua tay | Giam `Bo ga theo cua` |

Bang so lieu duoi preview tu canh bao khi vuot muc tieu (mat vach > 2%, cte_rms > 0.15, lai doi dau > 3 lan/giay).

## 4. Cho xe chay that

**Lan dau: ke banh khoi mat dat.** Chay cell duoi, roi bam **2. ARM**.

Giao dien tu tu choi ARM neu dang mat vach tren 20% frame. Nut **DUNG KHAN CAP** cat ga ngay. Camera dung hoac loi xu ly cung tu dong DISARM.

In [ ]:
# Dong giao dien cu truoc de tra camera lai
try:
    ui.close()
except NameError:
    pass

from tools.tune_lane_jupyter import launch
ui = launch(driver_kind='nvidia')

## 5. Chay luot day du bang config vua luu

Giao dien tune KHONG ghi log CSV. So lieu chinh thuc phai lay tu mot luot chay day du qua CLI - do moi la con so dua vao Technical Paper.

In [ ]:
!python3 -m src.jetracer_baseline.cli run --task speed --driver nvidia \n    --override configs/tuned.yaml --max-seconds 60 --record

## 6. Dong truoc khi tat notebook

In [ ]:
ui.close()